# NABirds -> ResNet-50 Fine-Tuning (Head Only)
This notebook builds a species-level dataset from NABirds and fine-tunes only the final classification layer of ResNet-50.

What this does:
- Uses your target species list.
- Auto-normalizes name differences (e.g., `Grey`/`Gray`, hyphenation, plural forms).
- Merges all NABirds subsets/variants (adult/juvenile, male/female, morphs) into one species label.
- Crops each image to NABirds bounding box.
- Resizes with aspect ratio preserved and pads to `240x240` using ImageNet mean color.
- Uses NABirds official train/test split and creates a validation split from train.


In [117]:
from pathlib import Path
import os
import re
import random
import time
from collections import defaultdict

import numpy as np
import pandas as pd
from PIL import Image, ImageOps

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from tqdm.auto import tqdm

try:
    import psutil
except ImportError:
    psutil = None



In [118]:
# Paths and reproducibility
DATA_ROOT = Path("NABirds Dataset/nabirds")
IMAGES_DIR = DATA_ROOT / "images"

SEED = 42
VAL_FRACTION = 0.10
BATCH_SIZE = 32
DEFAULT_NUM_WORKERS = 4

try:
    from IPython import get_ipython
    IN_NOTEBOOK = get_ipython() is not None
except Exception:
    IN_NOTEBOOK = False

# Notebook + spawn multiprocessing often fails to pickle custom Dataset classes.
NUM_WORKERS = 0 if IN_NOTEBOOK else DEFAULT_NUM_WORKERS
NUM_EPOCHS = 8
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 2e-4
TARGET_SIZE = 240
LABEL_SMOOTHING = 0.00

# Split file config
DEFAULT_TRAIN_TEST_SPLIT_FILE = "train_test_split.txt"
TRAIN_TEST_SPLIT_FILE = "train_test_split_8020_target_species.txt"

# Defaults used for checkpoint naming tags
DEFAULT_WEIGHT_DECAY = 1e-4

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print(f"Using device: {DEVICE}")
print(f"IN_NOTEBOOK={IN_NOTEBOOK}, NUM_WORKERS={NUM_WORKERS}")


Using device: mps
IN_NOTEBOOK=True, NUM_WORKERS=0


In [119]:
# Target species list (from your original notebook)
TARGET_SPECIES = [
    "American Robin",
    "Mourning Dove",
    "Blue Jay",
    "House Sparrow",
    "Canada Goose",
    "Northern Mockingbird",
    "American Crow",
    "Carolina Chickadee",
    "Red-tailed Hawk",
    "Northern Cardinal",
    "Killdeer",
    "Dark-eyed Junco",
    "White-breasted Nuthatch",
    "Bald Eagle",
    "Red-bellied Woodpecker",
    "Carolina Wren",
    "Song Sparrow",
    "Wild Turkey",
    "European Starling",
    "Tufted Titmouse",
    "Mallard",
    "Common Raven",
    "American Goldfinch",
    "House Wren",
    "Eastern Phoebe",
    "Common Yellowthroat",
    "Great Horned Owl",
    "Northern Flicker",
    "Grey Catbird",
    "White-throated Sparrow",
    "Chimney Swift",
    "Belted Kingfisher",
    "Red-winged Blackbird",
    "Laughing Gull",
    "House Finch",
    "Common Loon",
    "Great Crested Flycatcher",
    "Ruby-crowned Kinglet",
    "Hermit Thrush",
    "Wood Duck",
    "Yellow Warbler",
    "Chipping Sparrow",
    "Red-eyed Vireo",
    "Tree Swallow",
    "Cooper's Hawk",
    "Ovenbird",
    "Winter Wren",
    "Cedar Waxwing",
    "Snow Goose",
    "Brown Thrasher",
    "Eastern Screech Owl",
    "Downy Woodpecker",
    "Rock Pigeon",
    "Wood Thrush",
    "Red-breasted Nuthatch",
    "Common Grackle",
    "Eastern Wood-Pewee",
    "Pileated Woodpecker",
    "Brown-headed Cowbird",
    "American Woodcock",
    "Barred Owl",
    "Golden-crowned Kinglet",
    "Eastern Whip-poor-will",
    "Indigo Bunting",
    "Brown Creeper",
    "Fish Crow",
    "Barn Swallow",
    "Eastern Towhee",
    "Warbling Vireo",
    "Ruby-throated Hummingbird",
    "Field Sparrow",
    "Eastern Bluebird",
    "Hairy Woodpecker",
    "Baltimore Orioles",
    "Eastern Meadowlark",
    "Black-capped Chickadee",
    "Osprey",
    "Scarlet Tanager",
    "Eastern Kingbird",
    "Great Blue Heron",
    "Yellow-billed Cuckoo",
    "Red-headed Woodpecker",
    "Rose-breasted Grosbeak",
    "Yellow-bellied Sapsucker",
    "Black-and-white Warbler",
    "Willow Flycatcher",
    "Hooded Merganser",
    "American Redstart",
    "Green Heron",
    "Purple Martin",
    "Yellow-rumped Warbler",
    "American Kestrel",
    "Common Nighthawk",
    "Ruffed Grouse",
    "Common Merganser",
    "Great Egret",
    "Double-crested Cormorant",
    "Mute Swan",
    "Turkey Vulture",
    "Black Vulture",
]

len(TARGET_SPECIES)


100

## Build species-level mapping
NABirds includes variant classes. We map each target species to all matching NABirds class IDs after name normalization, then merge them into one label.


In [120]:
def canonicalize_name(name: str) -> str:
    # Remove variant suffixes like "(Adult male)", then normalize punctuation and spacing.
    name = re.sub(r"\s*\([^)]*\)\s*", " ", name)
    name = name.lower()
    name = name.replace("grey", "gray")
    name = name.replace("orioles", "oriole")
    name = name.replace("-", " ")
    name = name.replace("'", "")
    name = re.sub(r"[^a-z0-9 ]+", " ", name)
    name = re.sub(r"\s+", " ", name).strip()
    return name

# Load NABirds metadata
images = pd.read_csv(DATA_ROOT / "images.txt", sep=" ", names=["image_id", "image_rel_path"])
labels = pd.read_csv(DATA_ROOT / "image_class_labels.txt", sep=" ", names=["image_id", "class_id"])
splits = pd.read_csv(DATA_ROOT / TRAIN_TEST_SPLIT_FILE, sep=" ", names=["image_id", "is_train"])
bboxes = pd.read_csv(DATA_ROOT / "bounding_boxes.txt", sep=" ", names=["image_id", "x", "y", "w", "h"])
# Parse classes.txt robustly: first token is class_id, remainder is class_name.
class_rows = []
with open(DATA_ROOT / "classes.txt", "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        cid, cname = line.split(maxsplit=1)
        class_rows.append((int(cid), cname))
classes = pd.DataFrame(class_rows, columns=["class_id", "class_name"])

valid_class_ids = set(labels["class_id"].unique())
classes = classes[classes["class_id"].isin(valid_class_ids)].copy()
classes["canon"] = classes["class_name"].map(canonicalize_name)

species_to_class_ids = {}
dropped_species = []
normalization_report = {}

for species in TARGET_SPECIES:
    canon = canonicalize_name(species)
    matched = classes.loc[classes["canon"] == canon, "class_id"].tolist()
    if matched:
        species_to_class_ids[species] = sorted(set(matched))
        # Show auto-normalized cases for transparency.
        canonical_forms = sorted(set(classes.loc[classes["canon"] == canon, "class_name"]))
        if species not in canonical_forms:
            normalization_report[species] = canonical_forms
    else:
        dropped_species.append(species)

print(f"Requested species: {len(TARGET_SPECIES)}")
print(f"Mapped species:    {len(species_to_class_ids)}")
print(f"Dropped species:   {len(dropped_species)}")
if dropped_species:
    print("Dropped (not in NABirds):", dropped_species)

if normalization_report:
    print("\nAuto-normalized names:")
    for src, matched_names in normalization_report.items():
        print(f"- {src} -> {matched_names[0]}")


Requested species: 100
Mapped species:    98
Dropped species:   2
Dropped (not in NABirds): ['Eastern Whip-poor-will', 'Willow Flycatcher']

Auto-normalized names:
- American Robin -> American Robin (Adult)
- House Sparrow -> House Sparrow (Female/Juvenile)
- Red-tailed Hawk -> Red-tailed Hawk (Dark morph)
- Northern Cardinal -> Northern Cardinal (Adult Male)
- Dark-eyed Junco -> Dark-eyed Junco (Oregon)
- Bald Eagle -> Bald Eagle (Adult, subadult)
- European Starling -> European Starling (Breeding Adult)
- Mallard -> Mallard (Breeding male)
- American Goldfinch -> American Goldfinch (Breeding Male)
- Common Yellowthroat -> Common Yellowthroat (Adult Male)
- Northern Flicker -> Northern Flicker (Red-shafted)
- Grey Catbird -> Gray Catbird
- White-throated Sparrow -> White-throated Sparrow (Tan-striped/immature)
- Red-winged Blackbird -> Red-winged Blackbird (Female/juvenile)
- Laughing Gull -> Laughing Gull (Breeding)
- House Finch -> House Finch (Adult Male)
- Common Loon -> Common Lo

In [121]:
# Build a class_id -> final species index map (merging all variants into one label)
final_species = sorted(species_to_class_ids.keys())
species_to_idx = {name: i for i, name in enumerate(final_species)}

class_id_to_species_idx = {}
for species, class_ids in species_to_class_ids.items():
    y = species_to_idx[species]
    for cid in class_ids:
        class_id_to_species_idx[cid] = y

print(f"Final class count: {len(final_species)}")


Final class count: 98


## Assemble filtered sample table
This keeps only images belonging to your mapped species and attaches split + bounding box metadata.


In [122]:
df = images.merge(labels, on="image_id", how="inner")
df = df.merge(splits, on="image_id", how="inner")
df = df.merge(bboxes, on="image_id", how="inner")

df = df[df["class_id"].isin(class_id_to_species_idx)].copy()
df["target"] = df["class_id"].map(class_id_to_species_idx)
df["species"] = df["target"].map({v: k for k, v in species_to_idx.items()})
df["image_path"] = df["image_rel_path"].map(lambda p: str(IMAGES_DIR / p))

print("Filtered samples:", len(df))
print("Train/Test by official split:")
print(df["is_train"].value_counts().rename(index={1: "train", 0: "test"}))


Filtered samples: 14579
Train/Test by official split:
is_train
train    11663
test      2916
Name: count, dtype: int64


In [123]:
# Stratified validation split from official train split
# Normalize split dtype defensively (some pandas versions parse as object/string).
df = df.copy()
df["is_train"] = pd.to_numeric(df["is_train"], errors="coerce")
df = df[df["is_train"].isin([0, 1])].copy()
df["is_train"] = df["is_train"].astype(int)

train_df = df[df["is_train"] == 1].copy().reset_index(drop=True)
test_df = df[df["is_train"] == 0].copy().reset_index(drop=True)

if train_df.empty:
    raise RuntimeError("No training samples found after filtering. Check earlier mapping output and class matches.")

rng = np.random.default_rng(SEED)
val_indices = []

for _, group in train_df.groupby("target", sort=False):
    idxs = group.index.to_numpy(copy=True)
    rng.shuffle(idxs)

    if len(idxs) < 2:
        n_val = 0
    else:
        n_val = max(1, int(round(len(idxs) * VAL_FRACTION)))
        n_val = min(n_val, len(idxs) - 1)

    if n_val > 0:
        val_indices.extend(idxs[:n_val].tolist())

# Fallback: guarantee non-empty val split when possible.
if len(val_indices) == 0 and len(train_df) > 1:
    val_indices = [int(rng.integers(0, len(train_df)))]

val_mask = train_df.index.isin(val_indices)
val_df = train_df[val_mask].copy().reset_index(drop=True)
train_df = train_df[~val_mask].copy().reset_index(drop=True)

print(f"Train samples: {len(train_df)}")
print(f"Val samples:   {len(val_df)}")
print(f"Test samples:  {len(test_df)}")


Train samples: 10487
Val samples:   1176
Test samples:  2916


## Dataset and transforms
Each sample is cropped to the bird bounding box, resized with preserved aspect ratio, then padded to `240x240` using ImageNet mean color.


In [124]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
IMAGENET_PAD_RGB = tuple(int(round(c * 255)) for c in IMAGENET_MEAN)


def crop_resize_pad(img: Image.Image, bbox, size=240, pad_rgb=(124, 116, 104)) -> Image.Image:
    x, y, w, h = bbox
    x1 = max(0, int(np.floor(x)))
    y1 = max(0, int(np.floor(y)))
    x2 = min(img.width, int(np.ceil(x + w)))
    y2 = min(img.height, int(np.ceil(y + h)))

    if x2 <= x1 or y2 <= y1:
        cropped = img
    else:
        cropped = img.crop((x1, y1, x2, y2))

    scale = min(size / cropped.width, size / cropped.height)
    new_w = max(1, int(round(cropped.width * scale)))
    new_h = max(1, int(round(cropped.height * scale)))
    resized = cropped.resize((new_w, new_h), resample=Image.BILINEAR)

    pad_left = (size - new_w) // 2
    pad_top = (size - new_h) // 2
    pad_right = size - new_w - pad_left
    pad_bottom = size - new_h - pad_top

    return ImageOps.expand(resized, border=(pad_left, pad_top, pad_right, pad_bottom), fill=pad_rgb)


train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


class NABirdsSpeciesDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, transform=None):
        self.df = frame.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["image_path"]).convert("RGB")
        img = crop_resize_pad(
            img,
            bbox=(row["x"], row["y"], row["w"], row["h"]),
            size=TARGET_SIZE,
            pad_rgb=IMAGENET_PAD_RGB,
        )

        if self.transform is not None:
            img = self.transform(img)

        target = int(row["target"])
        return img, target


In [125]:
train_ds = NABirdsSpeciesDataset(train_df, transform=train_transform)
val_ds = NABirdsSpeciesDataset(val_df, transform=eval_transform)
test_ds = NABirdsSpeciesDataset(test_df, transform=eval_transform)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)
val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)
test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

print(f"Dataloaders ready: train={len(train_ds)}, val={len(val_ds)}, test={len(test_ds)}")


Dataloaders ready: train=10487, val=1176, test=2916


## ResNet-50 setup (freeze backbone)
All layers are frozen except the final `fc` layer.


In [126]:
num_classes = len(final_species)

weights = models.ResNet50_Weights.IMAGENET1K_V2
model = models.resnet50(weights=weights)

for p in model.parameters():
    p.requires_grad = False

in_features = model.fc.in_features
model.fc = nn.Linear(in_features, num_classes)

model = model.to(DEVICE)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable_params:,} / {total_params:,}")


Trainable params: 200,802 / 23,708,834


In [127]:
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
optimizer = torch.optim.AdamW(model.fc.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)


In [128]:
def fmt_gb(num_bytes: int) -> str:
    return f"{num_bytes / (1024 ** 3):.2f} GB"


def get_device_memory_label(device: torch.device) -> str:
    if device.type == "cuda":
        allocated = torch.cuda.memory_allocated(device)
        reserved = torch.cuda.memory_reserved(device)
        return f"alloc={fmt_gb(allocated)} reserved={fmt_gb(reserved)}"
    if device.type == "mps" and hasattr(torch, "mps"):
        # MPS memory telemetry is available on Apple Silicon builds.
        alloc = torch.mps.current_allocated_memory()
        driver = torch.mps.driver_allocated_memory()
        rec_max = torch.mps.recommended_max_memory()
        return f"alloc={fmt_gb(alloc)} driver={fmt_gb(driver)} max={fmt_gb(rec_max)}"
    return "n/a"


def run_epoch(model, loader, criterion, optimizer=None, split_name="train"):
    is_train = optimizer is not None
    model.train(is_train)

    proc = psutil.Process(os.getpid()) if psutil is not None else None

    running_loss = 0.0
    running_correct = 0
    running_total = 0

    epoch_start = time.perf_counter()
    pbar = tqdm(loader, leave=False, desc=f"{split_name}")

    for images, targets in pbar:
        step_start = time.perf_counter()

        images = images.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)

        if is_train:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(is_train):
            logits = model(images)
            loss = criterion(logits, targets)
            if is_train:
                loss.backward()
                optimizer.step()

        if DEVICE.type == "cuda":
            torch.cuda.synchronize(DEVICE)
        elif DEVICE.type == "mps" and hasattr(torch, "mps"):
            torch.mps.synchronize()

        preds = logits.argmax(dim=1)
        running_loss += loss.item() * targets.size(0)
        running_correct += (preds == targets).sum().item()
        running_total += targets.size(0)

        step_time = time.perf_counter() - step_start
        avg_loss_so_far = running_loss / max(1, running_total)
        avg_acc_so_far = running_correct / max(1, running_total)

        rss_label = fmt_gb(proc.memory_info().rss) if proc is not None else "n/a"
        device_mem_label = get_device_memory_label(DEVICE)

        pbar.set_postfix(
            loss=f"{avg_loss_so_far:.4f}",
            acc=f"{avg_acc_so_far:.4f}",
            step_s=f"{step_time:.3f}",
            rss=rss_label,
            dev_mem=device_mem_label,
        )

    avg_loss = running_loss / max(1, running_total)
    avg_acc = running_correct / max(1, running_total)
    epoch_time_s = time.perf_counter() - epoch_start
    return avg_loss, avg_acc, epoch_time_s


best_val_acc = -1.0
best_state = None
history = []

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss, train_acc, train_time = run_epoch(
        model, train_loader, criterion, optimizer=optimizer, split_name=f"train e{epoch:02d}"
    )
    val_loss, val_acc, val_time = run_epoch(
        model, val_loader, criterion, optimizer=None, split_name=f"val   e{epoch:02d}"
    )

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_acc": val_acc,
        "train_time_s": train_time,
        "val_time_s": val_time,
    })

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} train_time={train_time:.1f}s | "
        f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} val_time={val_time:.1f}s"
    )

if best_state is not None:
    model.load_state_dict(best_state)
    print(f"Loaded best checkpoint from training (best val_acc={best_val_acc:.4f})")

history_df = pd.DataFrame(history)
history_df



train e01:   0%|          | 0/328 [00:00<?, ?it/s]

val   e01:   0%|          | 0/37 [00:00<?, ?it/s]

Epoch 01 | train_loss=2.9790 train_acc=0.3928 train_time=97.2s | val_loss=2.0663 val_acc=0.5706 val_time=9.9s


train e02:   0%|          | 0/328 [00:00<?, ?it/s]

val   e02:   0%|          | 0/37 [00:00<?, ?it/s]

...

Epoch 07 | train_loss=0.4528 train_acc=0.9237 train_time=134.5s | val_loss=0.8621 val_acc=0.7755 val_time=13.5s


train e08:   0%|          | 0/328 [00:00<?, ?it/s]

val   e08:   0%|          | 0/37 [00:00<?, ?it/s]

Epoch 08 | train_loss=0.3851 train_acc=0.9413 train_time=135.7s | val_loss=0.8246 val_acc=0.7806 val_time=13.8s
Loaded best checkpoint from training (best val_acc=0.7806)


,epoch,train_loss,train_acc,val_loss,val_acc,train_time_s,val_time_s
0,1,2.979020,0.392772,2.066332,0.570578,97.216201,9.939246
1,2,1.545993,0.707161,1.497291,0.675170,95.695452,10.047767
2,3,1.061397,0.804234,1.192683,0.731293,106.608913,9.726493
3,4,0.806935,0.848860,1.066144,0.744048,94.871015,10.149344
4,5,0.653395,0.883761,0.980893,0.764456,99.118680,9.933871
5,6,0.546562,0.905121,0.905749,0.772959,114.577895,13.691463
6,7,0.452838,0.923715,0.862053,0.775510,134.504317,13.519505
7,8,0.385130,0.941261,0.824589,0.780612,135.718501,13.847376


In [129]:
test_loss, test_acc, test_time = run_epoch(
    model, test_loader, criterion, optimizer=None, split_name="test"
)
print(f"Test loss: {test_loss:.4f}")
print(f"Test acc:  {test_acc:.4f}")
print(f"Test time: {test_time:.1f}s")



test:   0%|          | 0/92 [00:00<?, ?it/s]

Test loss: 0.7785
Test acc:  0.8069
Test time: 34.0s


In [130]:
# Optional: save class names and model weights
OUTPUT_DIR = Path("artifacts")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def format_float_tag(x: float) -> str:
    # Use compact scientific notation, e.g. 2e-4.
    return f"{x:.0e}".replace("e-0", "e-").replace("e+0", "e+")


def split_ratio_tag(split_filename: str) -> str:
    # Create a short tag from the split filename.
    # Example: train_test_split_8020_target_species.txt -> 80-20
    name = Path(split_filename).stem.lower()
    m = re.search(r"(\d{2})(\d{2})", name)
    if m:
        a, b = int(m.group(1)), int(m.group(2))
        if a + b == 100:
            return f"{a}-{b}"
    return Path(split_filename).stem


ckpt_tags = []
if WEIGHT_DECAY != DEFAULT_WEIGHT_DECAY:
    ckpt_tags.append(f"decay_{format_float_tag(WEIGHT_DECAY)}")
if TRAIN_TEST_SPLIT_FILE != DEFAULT_TRAIN_TEST_SPLIT_FILE:
    ckpt_tags.append(f"tt_{split_ratio_tag(TRAIN_TEST_SPLIT_FILE)}")

CKPT_TAG_SUFFIX = ("_" + "_".join(ckpt_tags)) if ckpt_tags else ""

STAGE1_CKPT_NAME = f"resnet50_nabirds_head_only{CKPT_TAG_SUFFIX}.pt"
STAGE2_CKPT_NAME = f"resnet50_nabirds_layer4_finetuned{CKPT_TAG_SUFFIX}.pt"

stage1_ckpt_path = OUTPUT_DIR / STAGE1_CKPT_NAME
torch.save(model.state_dict(), stage1_ckpt_path)
pd.Series(final_species).to_csv(OUTPUT_DIR / "label_names.csv", index=False, header=["species"])
print("Saved:", stage1_ckpt_path)
print("Saved:", OUTPUT_DIR / "label_names.csv")
print("Checkpoint tag suffix:", CKPT_TAG_SUFFIX or "(default)")


Saved: artifacts/resnet50_nabirds_head_only_decay_2e-4_tt_80-20.pt
Saved: artifacts/label_names.csv
Checkpoint tag suffix: _decay_2e-4_tt_80-20


## Stage 2: Unfreeze `layer4` + `fc` and fine-tune
This stage starts from the saved head-only checkpoint in `artifacts/` and fine-tunes the second-to-last ResNet block (`layer4`) plus the classifier (`fc`).


In [131]:
# Stage-2 hyperparameters
STAGE2_EPOCHS = 5
STAGE2_LR = 2e-4
STAGE2_WEIGHT_DECAY = 2e-4

stage2_ckpt_path = OUTPUT_DIR / STAGE1_CKPT_NAME
if not stage2_ckpt_path.exists():
    raise FileNotFoundError(f"Missing checkpoint: {stage2_ckpt_path}. Run Stage 1 save cell first.")


In [132]:
# Rebuild model, load Stage-1 weights, then unfreeze layer4 + fc only.
stage2_model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
stage2_model.fc = nn.Linear(stage2_model.fc.in_features, len(final_species))

state = torch.load(stage2_ckpt_path, map_location="cpu")
stage2_model.load_state_dict(state)

for p in stage2_model.parameters():
    p.requires_grad = False

for p in stage2_model.layer4.parameters():
    p.requires_grad = True
for p in stage2_model.fc.parameters():
    p.requires_grad = True

stage2_model = stage2_model.to(DEVICE)

stage2_trainable = [p for p in stage2_model.parameters() if p.requires_grad]
stage2_optimizer = torch.optim.AdamW(
    stage2_trainable,
    lr=STAGE2_LR,
    weight_decay=STAGE2_WEIGHT_DECAY,
)

trainable_params = sum(p.numel() for p in stage2_model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in stage2_model.parameters())
print(f"Stage 2 trainable params: {trainable_params:,} / {total_params:,}")


Stage 2 trainable params: 15,165,538 / 23,708,834


In [133]:
# Stage-2 training loop
best_val_acc_stage2 = -1.0
best_state_stage2 = None
history_stage2 = []

for epoch in range(1, STAGE2_EPOCHS + 1):
    train_loss, train_acc, train_time = run_epoch(
        stage2_model,
        train_loader,
        criterion,
        optimizer=stage2_optimizer,
        split_name=f"s2 train e{epoch:02d}",
    )
    val_loss, val_acc, val_time = run_epoch(
        stage2_model,
        val_loader,
        criterion,
        optimizer=None,
        split_name=f"s2 val   e{epoch:02d}",
    )

    history_stage2.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_acc": val_acc,
        "train_time_s": train_time,
        "val_time_s": val_time,
    })

    if val_acc > best_val_acc_stage2:
        best_val_acc_stage2 = val_acc
        best_state_stage2 = {k: v.cpu().clone() for k, v in stage2_model.state_dict().items()}

    print(
        f"Stage2 Epoch {epoch:02d} | "
        f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} train_time={train_time:.1f}s | "
        f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} val_time={val_time:.1f}s"
    )

if best_state_stage2 is not None:
    stage2_model.load_state_dict(best_state_stage2)
    print(f"Loaded best Stage-2 checkpoint (best val_acc={best_val_acc_stage2:.4f})")

history_stage2_df = pd.DataFrame(history_stage2)
history_stage2_df


s2 train e01:   0%|          | 0/328 [00:00<?, ?it/s]

s2 val   e01:   0%|          | 0/37 [00:00<?, ?it/s]

Stage2 Epoch 01 | train_loss=0.1811 train_acc=0.9500 train_time=166.3s | val_loss=0.4164 val_acc=0.8784 val_time=13.6s


s2 train e02:   0%|          | 0/328 [00:00<?, ?it/s]

s2 val   e02:   0%|          | 0/37 [00:00<?, ?it/s]

...

Stage2 Epoch 04 | train_loss=0.0144 train_acc=0.9971 train_time=164.8s | val_loss=0.3918 val_acc=0.8980 val_time=13.3s


s2 train e05:   0%|          | 0/328 [00:00<?, ?it/s]

s2 val   e05:   0%|          | 0/37 [00:00<?, ?it/s]

Stage2 Epoch 05 | train_loss=0.0175 train_acc=0.9966 train_time=165.4s | val_loss=0.4171 val_acc=0.9022 val_time=14.1s
Loaded best Stage-2 checkpoint (best val_acc=0.9022)


,epoch,train_loss,train_acc,val_loss,val_acc,train_time_s,val_time_s
0,1,0.181095,0.950033,0.416379,0.878401,166.331288,13.631518
1,2,0.043315,0.989606,0.383690,0.888605,168.859969,13.721703
2,3,0.023114,0.995137,0.375636,0.886054,168.024721,13.634977
3,4,0.014438,0.997139,0.391835,0.897959,164.828076,13.310206
4,5,0.017500,0.996567,0.417130,0.902211,165.413841,14.074078


In [134]:
# Stage-2 test evaluation + save
test_loss_s2, test_acc_s2, test_time_s2 = run_epoch(
    stage2_model,
    test_loader,
    criterion,
    optimizer=None,
    split_name="s2 test",
)
print(f"Stage 2 test loss: {test_loss_s2:.4f}")
print(f"Stage 2 test acc:  {test_acc_s2:.4f}")
print(f"Stage 2 test time: {test_time_s2:.1f}s")

stage2_ckpt_out = OUTPUT_DIR / STAGE2_CKPT_NAME
torch.save(stage2_model.state_dict(), stage2_ckpt_out)
print("Saved:", stage2_ckpt_out)


s2 test:   0%|          | 0/92 [00:00<?, ?it/s]

Stage 2 test loss: 0.3562
Stage 2 test acc:  0.9119
Stage 2 test time: 35.4s
Saved: artifacts/resnet50_nabirds_layer4_finetuned_decay_2e-4_tt_80-20.pt
